# Day 13 – Advanced RAG – Extensive Solutions

Complete solutions for multi‑query retrieval, HyDE, reranking, and sub‑query decomposition.

## Setup: Load documents and base retriever

In [ ]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.chat_models import ChatOpenAI
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain.retrievers.multi_query import MultiQueryRetriever
import openai
from dotenv import load_dotenv
import os

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

# Sample corpus
documents = [
    "The Eiffel Tower is in Paris, France.",
    "Paris is the capital of France.",
    "The Louvre Museum is also in Paris.",
    "Jupiter is the largest planet in our solar system.",
    "Mars is called the Red Planet.",
]
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = splitter.create_documents(documents)

embeddings = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(chunks, embeddings)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

print("Setup complete.")

## Exercise 1: Precision comparison (naive vs multi‑query vs HyDE)

We'll implement each method and evaluate recall@3 on a small test set.

In [ ]:
# Test queries with known relevant document indices (ground truth)
test_queries = [
    ("What is the famous tower in Paris?", [0]),
    ("Capital of France", [1]),
    ("Largest planet", [3]),
]

def naive_retrieve(query, retriever, k=3):
    docs = retriever.get_relevant_documents(query)
    return [doc.page_content for doc in docs[:k]]

# Multi-query retriever (LangChain built-in)
multi_retriever = MultiQueryRetriever.from_llm(retriever=base_retriever, llm=llm)

def multi_query_retrieve(query, k=3):
    docs = multi_retriever.get_relevant_documents(query)
    return [doc.page_content for doc in docs[:k]]

# HyDE implementation
def hyde_retrieve(query, llm, vectorstore, k=3):
    # Generate hypothetical document
    hypo_prompt = f"Write a short passage that answers the question: {query}"
    hypothetical = llm.predict(hypo_prompt)
    # Use hypothetical document as query for similarity search
    docs = vectorstore.similarity_search(hypothetical, k=k)
    return [doc.page_content for doc in docs]

# Evaluation function
def recall_at_k(retrieved_docs, relevant_indices, corpus):
    retrieved_texts = retrieved_docs
    relevant_texts = [corpus[i] for i in relevant_indices]
    matches = sum(1 for r in retrieved_texts if r in relevant_texts)
    return matches / len(relevant_indices)

corpus_texts = [doc.page_content for doc in chunks]
methods = {
    "Naive": lambda q: naive_retrieve(q, base_retriever, 3),
    "MultiQuery": lambda q: multi_query_retrieve(q, 3),
    "HyDE": lambda q: hyde_retrieve(q, llm, vectorstore, 3)
}

for name, method in methods.items():
    total_recall = 0
    for query, relevant in test_queries:
        retrieved = method(query)
        recall = recall_at_k(retrieved, relevant, corpus_texts)
        total_recall += recall
    avg_recall = total_recall / len(test_queries)
    print(f"{name} average recall@3: {avg_recall:.2f}")

## Exercise 2: HyDE with different hypothetical document lengths

We'll generate 20, 100, and 300 word hypothetical answers and compare.

In [ ]:
def hyde_with_length(query, llm, vectorstore, length_words, k=3):
    prompt = f"Write a {length_words}-word passage answering: {query}"
    hypothetical = llm.predict(prompt)
    print(f"Hypothetical ({length_words} words): {hypothetical[:100]}...")
    docs = vectorstore.similarity_search(hypothetical, k=k)
    return [doc.page_content for doc in docs]

query = "What is the capital of France?"
for length in [20, 100, 300]:
    retrieved = hyde_with_length(query, llm, vectorstore, length, k=3)
    print(f"Length {length} retrieved: {retrieved}\n")

## Exercise 3: Cohere Rerank vs local cross‑encoder

We'll implement both and compare top‑3 results.

In [ ]:
# First, install required libraries
# !pip install cohere sentence-transformers

import cohere
from sentence_transformers import CrossEncoder

# Local cross-encoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def local_rerank(query, docs, top_n=3):
    pairs = [[query, doc.page_content] for doc in docs]
    scores = cross_encoder.predict(pairs)
    scored = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, score in scored[:top_n]]

# Cohere rerank (requires API key)
co = cohere.Client(os.getenv("COHERE_API_KEY"))

def cohere_rerank(query, docs, top_n=3):
    results = co.rerank(query=query, documents=[doc.page_content for doc in docs], top_n=top_n)
    return [docs[r.index] for r in results.results]

# Get initial retrieval (top 10)
query = "What is the famous tower in Paris?"
initial_docs = base_retriever.get_relevant_documents(query)  # returns 5 by default
print(f"Initial retrieved ({len(initial_docs)} docs):")
for d in initial_docs:
    print(f" - {d.page_content}")

# Apply rerankers
local_reranked = local_rerank(query, initial_docs, top_n=3)
print("\nLocal cross-encoder top 3:")
for d in local_reranked:
    print(f" - {d.page_content}")

# Uncomment if you have Cohere key
# cohere_reranked = cohere_rerank(query, initial_docs, top_n=3)
# print("\nCohere rerank top 3:")
# for d in cohere_reranked:
#     print(f" - {d.page_content}")

## Exercise 4: Sub‑query decomposition

Break a complex question into sub‑queries, retrieve for each, then synthesise.

In [ ]:
def decompose_query(complex_question, llm):
    prompt = f"""Break the following question into 2-3 simpler sub-questions.
    Question: {complex_question}
    Output each sub-question on a new line.
    """
    response = llm.predict(prompt)
    sub_questions = [q.strip() for q in response.split('\n') if q.strip()]
    return sub_questions

def subquery_retrieve(complex_question, llm, vectorstore, k=2):
    sub_queries = decompose_query(complex_question, llm)
    print(f"Sub-queries: {sub_queries}")
    all_docs = []
    for sq in sub_queries:
        docs = vectorstore.similarity_search(sq, k=k)
        all_docs.extend(docs)
    # Remove duplicates while preserving order
    seen = set()
    unique_docs = []
    for doc in all_docs:
        if doc.page_content not in seen:
            seen.add(doc.page_content)
            unique_docs.append(doc)
    return unique_docs

complex_q = "Compare the height of the Eiffel Tower and the largest planet's diameter."
retrieved = subquery_retrieve(complex_q, llm, vectorstore, k=2)
print("\nRetrieved documents:")
for d in retrieved:
    print(f" - {d.page_content}")